# UMAP on Petrophysical Data
## Machine learning for energy applications — Module 4, Exercise 3

**Uniform Manifold Approximation and Projection (UMAP)** is a non-linear
dimensionality-reduction technique that builds a weighted neighbourhood graph of the data
and then optimizes a low-dimensional layout that approximates that graph. It scales well to
large datasets and, unlike scikit-learn's t-SNE, it can transform new rows. Claims that it
preserves global structure better than t-SNE are debated: much of the difference comes from
the initialization.

In this exercise we:
1. Load petrophysical well-log data with facies labels
2. Apply UMAP and visualize the 2D embedding
3. Experiment with `n_neighbors` and `min_dist` hyperparameters
4. Compare UMAP with t-SNE and PCA, scoring every map with trustworthiness at k = 10

> **Note:** UMAP requires the optional `umap-learn` package, listed in
> `requirements-labs-optional.txt` at the course root (`pip install umap-learn`).
> We wrap the import in a try/except block, so the UMAP cells are skipped if it is missing.

> **Exploratory analysis.** The facies labels are kept out of the inputs: they only colour the
> maps after fitting. The scaler and every map are fitted on all 1,000 rows, so the maps describe
> this table. Islands in a map are not evidence that the facies are separable in the logs until
> that is checked in the logs themselves.

---
## 0 — Imports and Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.manifold import TSNE, trustworthiness
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Try importing UMAP — optional package: pip install umap-learn
# (Without ipywidgets, importing umap may print a harmless TqdmWarning about IProgress.)
try:
    import umap
    UMAP_AVAILABLE = True
    print("umap-learn is available.")
except ImportError:
    UMAP_AVAILABLE = False
    print("WARNING: umap-learn is NOT installed.")
    print("Install it with: pip install umap-learn")
    print("UMAP cells will be skipped; PCA and t-SNE will still run.")

plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 12,
})

RANDOM_STATE = 42
print("Setup complete.")

---
## 1 — Load and Explore the Data

The dataset contains synthetic petrophysical well-log measurements:
- **log_0 through log_9** — ten well-log channels
- **facies** — lithofacies label (integer class 0, 1 or 2; the file gives no facies names)

In [ ]:
df = pd.read_csv("petro.csv")
print(f"Shape: {df.shape}")
df.head(10)

In [ ]:
df.describe()

In [ ]:
print("Facies distribution:")
print(df["facies"].value_counts().sort_index())

### 1.1 Separate Features and Labels

The facies label is kept out of the feature matrix `X`: UMAP, t-SNE and PCA are fitted on the
ten logs only, and the labels are used afterwards to colour the maps.

In [ ]:
feature_cols = [c for c in df.columns if c != "facies"]
X = df[feature_cols].values
y = df["facies"].values          # used only to colour the maps after fitting
assert "facies" not in feature_cols

n_classes = len(np.unique(y))
print(f"Features : {len(feature_cols)}")
print(f"Samples  : {X.shape[0]}")
print(f"Classes  : {n_classes} ({np.unique(y)})")

---
## 2 — Standardize Features

The scaler is fitted on all rows because the maps are exploratory. If a UMAP map were reused to
place new wells, the scaler and UMAP would be fitted on training wells only and then applied to
the new wells with `transform`.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Means after scaling: {X_scaled.mean(axis=0).round(6)}")
print(f"Stds  after scaling: {X_scaled.std(axis=0).round(6)}")

---
## 3 — Apply UMAP

UMAP has two key hyperparameters:
- **n_neighbors** — controls local vs global structure (similar to perplexity in t-SNE)
- **min_dist** — controls how tightly points are packed in the embedding

We start with default values: `n_neighbors=15`, `min_dist=0.1`. Setting `random_state` makes
the run repeatable but forces single-threaded fitting, so we also pass `n_jobs=1` explicitly.
Every map is scored with trustworthiness at k = 10.

In [ ]:
if UMAP_AVAILABLE:
    reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1,
                        random_state=RANDOM_STATE, n_jobs=1)
    X_umap = reducer.fit_transform(X_scaled)
    print(f"UMAP output shape: {X_umap.shape}")
    print(f"Trustworthiness (k=10): {trustworthiness(X_scaled, X_umap, n_neighbors=10):.3f}")
else:
    print("Skipping UMAP — not installed. See instructions above.")
    X_umap = None

In [ ]:
if X_umap is not None:
    fig, ax = plt.subplots(figsize=(9, 7))
    scatter = ax.scatter(X_umap[:, 0], X_umap[:, 1],
                         c=y, cmap="tab10", alpha=0.7, s=20,
                         edgecolors="w", linewidth=0.3)
    cbar = plt.colorbar(scatter, ax=ax, ticks=range(n_classes))
    cbar.set_label("Facies")
    ax.set_xlabel("UMAP Dimension 1")
    ax.set_ylabel("UMAP Dimension 2")
    ax.set_title("UMAP Embedding (n_neighbors=15, min_dist=0.1, coloured by Facies)")
    plt.tight_layout()
    plt.show()
else:
    print("UMAP not available — skipping plot.")

---
## 4 — Experiment with n_neighbors

- **Small n_neighbors** (e.g., 5): emphasizes very local structure, can fragment clusters
- **Large n_neighbors** (e.g., 50): captures more global structure, smoother embedding

In [ ]:
if UMAP_AVAILABLE:
    nn_values = [5, 15, 50]
    umap_nn_results = {}

    for nn in nn_values:
        reducer = umap.UMAP(n_components=2, n_neighbors=nn, min_dist=0.1,
                            random_state=RANDOM_STATE, n_jobs=1)
        umap_nn_results[nn] = reducer.fit_transform(X_scaled)
        t = trustworthiness(X_scaled, umap_nn_results[nn], n_neighbors=10)
        print(f"n_neighbors={nn:>2d}  trustworthiness(k=10)={t:.3f}")
else:
    print("UMAP not available — skipping.")
    umap_nn_results = {}

In [ ]:
if umap_nn_results:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    for ax, nn in zip(axes, nn_values):
        X_emb = umap_nn_results[nn]
        sc = ax.scatter(X_emb[:, 0], X_emb[:, 1],
                        c=y, cmap="tab10", alpha=0.7, s=15,
                        edgecolors="w", linewidth=0.2)
        ax.set_title(f"n_neighbors = {nn}", fontsize=13)
        ax.set_xlabel("UMAP Dim 1")
        ax.set_ylabel("UMAP Dim 2")

    plt.suptitle("UMAP — Effect of n_neighbors (coloured by Facies)", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("UMAP not available — skipping plot.")

---
## 5 — Experiment with min_dist

- **min_dist = 0.0**: points are packed as tightly as possible, producing very dense clusters
- **min_dist = 0.5**: more spread-out embedding, preserving more of the broad topological structure

In [ ]:
if UMAP_AVAILABLE:
    md_values = [0.0, 0.1, 0.5]
    umap_md_results = {}

    for md in md_values:
        reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=md,
                            random_state=RANDOM_STATE, n_jobs=1)
        umap_md_results[md] = reducer.fit_transform(X_scaled)
        t = trustworthiness(X_scaled, umap_md_results[md], n_neighbors=10)
        print(f"min_dist={md:.1f}  trustworthiness(k=10)={t:.3f}")
else:
    print("UMAP not available — skipping.")
    umap_md_results = {}

In [ ]:
if umap_md_results:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    for ax, md in zip(axes, md_values):
        X_emb = umap_md_results[md]
        sc = ax.scatter(X_emb[:, 0], X_emb[:, 1],
                        c=y, cmap="tab10", alpha=0.7, s=15,
                        edgecolors="w", linewidth=0.2)
        ax.set_title(f"min_dist = {md}", fontsize=13)
        ax.set_xlabel("UMAP Dim 1")
        ax.set_ylabel("UMAP Dim 2")

    plt.suptitle("UMAP — Effect of min_dist (coloured by Facies)", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("UMAP not available — skipping plot.")

---
## 6 — Compare UMAP with t-SNE and PCA

Let us apply all three methods and visualize side by side.

In [ ]:
# PCA
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)
print(f"PCA explained variance: {pca.explained_variance_ratio_.sum():.3f}")

# t-SNE
tsne = TSNE(n_components=2, perplexity=30, random_state=RANDOM_STATE, max_iter=1000)
X_tsne = tsne.fit_transform(X_scaled)
print(f"t-SNE KL divergence : {tsne.kl_divergence_:.4f}")

# Trustworthiness of every map on the same rows, at the same k
trust = {"PCA": trustworthiness(X_scaled, X_pca, n_neighbors=10),
         "t-SNE": trustworthiness(X_scaled, X_tsne, n_neighbors=10)}
if X_umap is not None:
    trust["UMAP"] = trustworthiness(X_scaled, X_umap, n_neighbors=10)
for name, t in trust.items():
    print(f"Trustworthiness (k=10) {name:<6s}: {t:.3f}")

In [ ]:
n_methods = 3 if X_umap is not None else 2
fig, axes = plt.subplots(1, n_methods, figsize=(7 * n_methods, 6))

# PCA
axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap="tab10", alpha=0.7, s=15,
                edgecolors="w", linewidth=0.2)
axes[0].set_title(f"PCA ({pca.explained_variance_ratio_.sum()*100:.1f}% var, T(10) = {trust['PCA']:.3f})")
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")

# t-SNE
axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1], c=y, cmap="tab10", alpha=0.7, s=15,
                edgecolors="w", linewidth=0.2)
axes[1].set_title(f"t-SNE (perplexity=30, T(10) = {trust['t-SNE']:.3f})")
axes[1].set_xlabel("t-SNE Dim 1")
axes[1].set_ylabel("t-SNE Dim 2")

# UMAP
if X_umap is not None:
    sc = axes[2].scatter(X_umap[:, 0], X_umap[:, 1], c=y, cmap="tab10", alpha=0.7, s=15,
                         edgecolors="w", linewidth=0.2)
    axes[2].set_title(f"UMAP (n_neighbors=15, T(10) = {trust['UMAP']:.3f})")
    axes[2].set_xlabel("UMAP Dim 1")
    axes[2].set_ylabel("UMAP Dim 2")

plt.suptitle("Dimensionality Reduction Comparison (coloured by Facies)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## Summary

In this exercise we applied UMAP to petrophysical data and covered:

1. **Data preparation** — separating features from facies labels and standardizing.
2. **UMAP embedding** — projecting 10 log channels into 2D for visualization.
3. **n_neighbors tuning** — small values emphasize local structure; large values let broader structure shape the layout.
4. **min_dist tuning** — controls cluster tightness in the embedding.
5. **Method comparison** — PCA (linear, fast, interpretable), t-SNE (non-linear, local focus, no transform for new rows), UMAP (non-linear, neighbourhood graph, can transform new rows), each scored with trustworthiness at k = 10.

**Key takeaway**: UMAP is a useful tool for visualizing high-dimensional petrophysical data,
and it can place new rows on an existing map. Like t-SNE, its axes are not interpretable and
its pictures depend on `n_neighbors`, `min_dist` and the seed. Well-separated islands in a map
coloured by facies are not proof that the facies are separable in the logs: report the settings
and the trustworthiness, repeat settings and seeds, and confirm any group in the original logs.